# 10 Figures
Publication figures, drawn only from files the earlier notebooks saved: the cached anatomy and tractograms of each
session (`data/cache/`) and the result tables (`outputs/results/`). Nothing is tracked, trained or scored here, so
the notebook can be re-run at any time to restyle the figures. Each figure is saved as PDF (vector, for the journal)
and PNG in `outputs/figures/`.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.patches import Patch
from retinopath.core import load_config
from retinopath import metrics as M, pipeline as PL

cfg = load_config(DEV_MODE)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

## Shared settings
Subjects, protocols, colours and the loaders used by every figure. A network is one method per variant: its
training seeds are averaged in the tables and pooled in the streamline counts.

In [ ]:
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42, "svg.fonttype": "none"})
subjects = json.load(open(cfg.P.out / "subjects.json")); test = subjects["test"]
clinical = [p for p in cfg.clinical_protocols if p in cfg.run_protocols]
PAIR_NAMES = list(M.PAIRS.values())
STYLE = {"reference": ("k", "reference (msmt_csd, full data)"), "proximity": ("0.7", "proximity only"), "csd": ("0.45", "classical CSD"),
         "voxel": ("C0", "network, voxel loss"), "path": ("C3", "network, path loss")}
OTHER = ["#e69f00", "#56b4e9", "#009e73", "#cc79a7", "#8c564b", "#bcbd22", "#17becf", "#9467bd", "#000080", "#f0e442"]
_extra_colours = {}
def colour(method):
    """Fixed colour per method: the five main ones from STYLE, the others in order of first use."""
    if method in STYLE: return STYLE[method][0]
    return _extra_colours.setdefault(method, OTHER[len(_extra_colours) % len(OTHER)])

def variant(method):
    """'net_path_s0' -> 'path'; classical names are kept."""
    return method[4:].rsplit("_s", 1)[0] if method.startswith("net_") else method

def table(name):
    path = RES / f"{name}.csv"
    if not path.exists(): return pd.DataFrame()
    scores = pd.read_csv(path); scores["variant"] = scores.method.map(variant) if "method" in scores else None; return scores

def anat_of(sid, kind="3T"):
    stored = np.load(cfg.P.cache / PL.session(sid, kind)["tag"] / "anat.npz")
    return {key: stored[key].item() if stored[key].ndim == 0 else stored[key] for key in stored.files}

def tracts(sid, name, method, kind="3T"):
    """Cached tractograms of one method; for a network variant, those of every training seed. Never computes."""
    session = PL.session(sid, kind)
    if method in STYLE and method not in ("reference", "proximity", "csd"):
        files = sorted(PL.sdir(cfg, session, "net_").glob(f"tract_{name}_net_{method}_s*.npz"))
    else:
        files = [PL.sdir(cfg, session, method) / f"tract_{name}_{method}.npz"]
    return [dict(np.load(f)) for f in files if f.exists()]

def save(fig, name):
    for extension in ("pdf", "png"): fig.savefig(FIG / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    print("saved", FIG / f"{name}.pdf")

def letter(axis, text, offset=-0.12):
    axis.text(offset, 1.06, text, transform=axis.transAxes, fontsize=10, fontweight="bold", va="bottom")

reference, classical, learned = table("h1_reference"), table("protocols"), table("learned")
# Main panels: the clinical-grade protocol on which classical CSD loses most RSI against the reference (median over
# subjects). Only classical results enter this choice, never the networks.
csd_loss = {p: reference[reference.method == "msmt_csd"].rsi_ree_all.median() - classical[(classical.method == "csd") & (classical.protocol == p)].rsi_ree_all.median()
            for p in clinical}
MAIN = max(clinical, key=lambda p: csd_loss[p] if np.isfinite(csd_loss[p]) else -np.inf)
print("test subjects:", len(test), "| clinical protocols:", clinical, "| RSI lost by CSD:", {p: round(v, 3) for p, v in csd_loss.items()},
      "| main panels:", MAIN)

## Like-to-like counts
For one area pair, streamline ends are binned by the retinotopic coordinate of each end (eccentricity or polar
angle), with bin edges at quantiles of the subject's valid V1 values (`prf.n_ecc_bins` bins, as for the binned AUC).
Each count is divided by what independent ends would give (row total × column total / total). A ratio above 1 on
the diagonal means ends at matching visual-field positions are joined more often than chance. This ratio removes
uneven coverage but not distance; the distance-matched RSI is the controlled measure.

In [ ]:
def binned_counts(anat, key, tractogram, code, variable):
    """Observed and expected (independent ends) counts, bins of `variable` ('ecc' or 'ang') of end a x end b."""
    n_bins = cfg.prf.n_ecc_bins; values = key[variable]
    valid_v1 = key["valid"] & (anat["area"] == 1)
    if valid_v1.sum() < n_bins: return np.zeros((n_bins, n_bins)), np.zeros((n_bins, n_bins))
    edges = np.quantile(values[valid_v1], np.linspace(0, 1, n_bins + 1))
    selected = (tractogram["pair"] == code) & key["valid"][tractogram["a"]] & key["valid"][tractogram["b"]]
    lower_bin, higher_bin = (np.clip(np.digitize(values[tractogram[end][selected]], edges[1:-1]), 0, n_bins - 1) for end in ("a", "b"))
    observed = np.zeros((n_bins, n_bins)); np.add.at(observed, (lower_bin, higher_bin), 1)
    expected = np.outer(observed.sum(1), observed.sum(0)) / max(observed.sum(), 1)
    return observed, expected

def pooled(method, name, code, variable, sids=test, kind="3T"):
    """Counts summed over subjects (and training seeds) -> ratio observed / expected; per-subject ratio by bin distance."""
    observed = expected = 0; profiles = []
    for sid in sids:
        anat = anat_of(sid, kind); key = M.get_key(anat, PL.cfg_for(cfg, PL.session(sid, kind)))
        subject_observed = subject_expected = 0
        for tractogram in tracts(sid, name, method, kind):
            tract_observed, tract_expected = binned_counts(anat, key, tractogram, code, variable); subject_observed, subject_expected = subject_observed + tract_observed, subject_expected + tract_expected
        if np.sum(subject_observed) == 0: continue
        observed, expected = observed + subject_observed, expected + subject_expected
        row, column = np.indices(subject_observed.shape); bin_distance = np.abs(row - column)
        profiles.append([subject_observed[bin_distance == offset].sum() / subject_expected[bin_distance == offset].sum() if subject_expected[bin_distance == offset].sum() else np.nan for offset in range(subject_observed.shape[0])])
    ratio = np.where(np.asarray(expected) > 0, np.asarray(observed) / np.where(np.asarray(expected) > 0, expected, 1), np.nan) if np.ndim(observed) else None
    return ratio, np.array(profiles)

def matrix(axis, ratio, title, pair, variable, vmax=1.0):
    n_bins = cfg.prf.n_ecc_bins
    if ratio is None:
        axis.text(0.5, 0.5, "no streamlines", ha="center", va="center", transform=axis.transAxes); axis.set_axis_off(); return None
    axis.set_facecolor("0.9")      # grey: no streamline end in that row or column
    image = axis.imshow(np.log2(np.maximum(ratio, 2.0 ** -vmax)), origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    lower_area, higher_area = pair.split("-"); label = "eccentricity" if variable == "ecc" else "polar angle"
    axis.set_xticks(range(n_bins), [str(index + 1) for index in range(n_bins)]); axis.set_yticks(range(n_bins), [str(index + 1) for index in range(n_bins)])
    axis.set_xlabel(f"{higher_area} {label} bin"); axis.set_ylabel(f"{lower_area} {label} bin"); axis.set_title(title)
    return image

def colorbar(fig, image, axis):
    """Colour bar beside a matrix panel, exactly as tall as the matrix."""
    bar = fig.colorbar(image, cax=axis.inset_axes([1.08, 0, 0.07, 1]), ticks=[-1, 0, 1]); bar.ax.set_yticklabels(["½×", "chance", "2×"]); bar.set_label("streamlines / expected")

## Figure 1: retinotopy and V1–V3 tractography of one subject
The first test subject, first hemisphere, seen from the medial side. (a) Visual areas, (b) polar angle and (c)
eccentricity from the subject's own pRF fit, on the very inflated surface: the whole hemisphere in (a), the boxed region
enlarged in (b, c). Without that file the white surface near V1–V3 is drawn instead. Vertices outside the pRF validity mask are dark grey. (d) The reference V1–V3 streamlines on the
white surface, coloured by the eccentricity of their lower-area end (V1, or V2 for V2–V3).

In [ ]:
from retinopath.subject import inflated_surface
sid = test[0]; anat = anat_of(sid); key = M.get_key(anat, cfg); hemi_index = 0; hemi = anat["hemis"][hemi_index]
roi = (anat["area"] > 0) & (anat["hemi"] == hemi_index); ecc_low, ecc_high = cfg.prf.ecc_min, cfg.prf.ecc_max
def camera(degrees):
    """Unit vector towards the viewer, on the medial side, turned `deg` towards the back; and the screen's x axis."""
    toward = np.array([np.cos(np.radians(degrees)) * (1 if hemi == "L" else -1), -np.sin(np.radians(degrees)), 0.0]); return toward, np.array([-toward[1], toward[0], 0.0])
inflated = inflated_surface(cfg, sid, hemi)
surface_points = inflated[anat["vert_id"]] if inflated is not None else anat["verts"]
if inflated is None: print("very inflated surface not downloaded (notebook 01): panels a-c use the white surface")

class View:
    """Orthographic view of the surface P within `radius` mm of V1-V3 (None: whole hemisphere): faces sorted back
    to front, Lambert shading."""
    def __init__(self, points, radius, degrees):
        toward, self.right = camera(degrees)
        near = (anat["hemi"] == hemi_index) & (True if radius is None else np.linalg.norm(points - points[roi].mean(0), axis=1) < radius)
        faces = anat["faces"][near[anat["faces"]].all(1)]; self.F = faces[np.argsort((points @ toward)[faces].mean(1))]
        face_normals = np.cross(points[self.F[:, 1]] - points[self.F[:, 0]], points[self.F[:, 2]] - points[self.F[:, 0]])
        self.shade = 0.55 + 0.45 * np.abs(face_normals @ toward) / (np.linalg.norm(face_normals, axis=1) + 1e-12)
        self.xy = np.c_[points @ self.right, points[:, 2]]; self.box = self.xy[np.unique(self.F)]
    def colors(self, values=None, cmap=None, vmin=0, vmax=1, mask=None):
        faces = self.F; face_colours = np.tile([0.82, 0.82, 0.82, 1.0], (len(faces), 1)); face_colours[roi[faces].all(1), :3] = 0.55
        if values is not None:
            selected = mask[faces].all(1); face_colours[selected] = plt.get_cmap(cmap)(np.clip((values[faces[selected, 0]] - vmin) / (vmax - vmin), 0, 1))
        face_colours[:, :3] *= self.shade[:, None]; return face_colours
    def zoom_box(self, pad=5.0):
        """(x0, x1, y0, y1) around V1-V3 on screen."""
        screen_points = self.xy[roi]; return screen_points[:, 0].min() - pad, screen_points[:, 0].max() + pad, screen_points[:, 1].min() - pad, screen_points[:, 1].max() + pad
    def draw(self, axis, colors, alpha=1.0, zoom=False):
        colors = colors.copy(); colors[:, 3] = alpha
        axis.add_collection(PolyCollection(self.xy[self.F], facecolors=colors, edgecolors="none"))
        x0, x1, y0, y1 = self.zoom_box() if zoom else (self.box[:, 0].min() - 2, self.box[:, 0].max() + 2, self.box[:, 1].min() - 2, self.box[:, 1].max() + 2)
        axis.set_xlim(x0, x1); axis.set_ylim(y0, y1)
        axis.set_aspect("equal"); axis.set_axis_off()

def cbar(axis, cmap, vmin, vmax, label, ticks):
    bar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax)), cax=axis.inset_axes([0.15, -0.08, 0.7, 0.04]),
                      orientation="horizontal", ticks=ticks)
    bar.set_label(label, labelpad=1)

inflated_view = View(surface_points, None, 15) if inflated is not None else View(surface_points, 40, 30); white_view = View(anat["verts"], 40, 30)
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.6)); fig.subplots_adjust(wspace=0.08, left=0.03, right=0.99, top=0.9, bottom=0.2)
AREA_COLOURS = ["#1b9e77", "#d95f02", "#7570b3"]
colours = inflated_view.colors(); in_roi = roi[inflated_view.F].all(1)
colours[in_roi, :3] = np.array([plt.matplotlib.colors.to_rgb(AREA_COLOURS[int(area_label) - 1]) for area_label in anat["area"][inflated_view.F[in_roi, 0]]]) * inflated_view.shade[in_roi, None]
inflated_view.draw(axes[0], colours)
zoom = inflated is not None                                            # whole inflated hemisphere in a, V1-V3 enlarged in b and c
if zoom:
    x0, x1, y0, y1 = inflated_view.zoom_box(); axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=0.6, color="k"))
    (left, right), (bottom, tractogram) = axes[0].get_xlim(), axes[0].get_ylim(); axes[0].set_xlim(min(left, x0 - 1), max(right, x1 + 1)); axes[0].set_ylim(min(bottom, y0 - 1), max(tractogram, y1 + 1))
axes[0].legend(handles=[Patch(color=AREA_COLOURS[position], label=f"V{position + 1}") for position in range(3)], loc="upper center",
                                 bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, handlelength=1, columnspacing=1)
inflated_view.draw(axes[1], inflated_view.colors(np.nan_to_num(key["ang"]), "hsv", 0, 360, key["valid"]), zoom=zoom); cbar(axes[1], "hsv", 0, 360, "polar angle (deg)", [0, 90, 180, 270, 360])
inflated_view.draw(axes[2], inflated_view.colors(np.nan_to_num(key["ecc"]), "viridis", ecc_low, ecc_high, key["valid"]), zoom=zoom); cbar(axes[2], "viridis", ecc_low, ecc_high, "eccentricity (deg)", np.linspace(ecc_low, ecc_high, 3).round(1))
tractogram = tracts(sid, "REF", "msmt_csd")[0]; streamlines = PL.streamlines_of(tractogram)
drawn = [index for index in range(len(streamlines)) if key["valid"][tractogram["a"][index]] and anat["hemi"][tractogram["a"][index]] == hemi_index]
drawn = np.random.default_rng(cfg.seed).permutation(drawn)[:1500]
white_view.draw(axes[3], white_view.colors(), alpha=0.35)
axes[3].add_collection(LineCollection([np.c_[streamlines[index] @ white_view.right, streamlines[index][:, 2]] for index in drawn], linewidths=0.4,
                                    colors=plt.get_cmap("viridis")(np.clip((key["ecc"][tractogram["a"][drawn]] - ecc_low) / (ecc_high - ecc_low), 0, 1))))
cbar(axes[3], "viridis", ecc_low, ecc_high, "eccentricity of lower-area end (deg)", np.linspace(ecc_low, ecc_high, 3).round(1))
for axis, letter_text, title in zip(axes, "abcd", ("visual areas", "polar angle", "eccentricity", "reference streamlines")):
    axis.set_title(title); axis.text(0.0, 1.02, letter_text, transform=axis.transAxes, fontsize=10, fontweight="bold", va="bottom")
save(fig, "fig1_retinotopy_tractography")
print("subject", sid, "hemisphere", hemi, "|", "very inflated" if inflated is not None else "white", "surface |", len(drawn), "streamlines drawn")

## Figure 2: retinotopy of the reference tractogram (H1)
(a–c) Eccentricity and (d–f) polar-angle like-to-like ratios of the reference tractogram, summed over test subjects.
(g, h) Ratio by distance between bins, one line per subject, reference against the proximity-only tractogram.
(i) RSI per area pair, one point per subject; lines join the same subject.

In [ ]:
fig = plt.figure(figsize=(7.2, 7.4)); gs = fig.add_gridspec(3, 3, hspace=0.6, wspace=0.75)
for row, variable in enumerate(("ecc", "ang")):
    for column, (code, pair) in enumerate(M.PAIRS.items()):
        axis = fig.add_subplot(gs[row, column]); ratio, _ = pooled("msmt_csd", "REF", code, variable)
        image = matrix(axis, ratio, pair, pair, variable); letter(axis, "abcdef"[3 * row + column], -0.35)
    if image is not None: colorbar(fig, image, axis)
bottom_grid = gs[2, :].subgridspec(1, 3, wspace=0.5)
for column, variable in enumerate(("ecc", "ang")):
    axis = fig.add_subplot(bottom_grid[column])
    for method, label in (("msmt_csd", "reference"), ("proximity", "proximity")):
        _, profiles = pooled(method, "REF", 12, variable)
        if len(profiles):
            axis.plot(profiles.T, color=colour(label), lw=0.6, alpha=0.5); axis.plot(np.nanmedian(profiles, 0), color=colour(label), lw=2, label=STYLE[label][1])
    axis.axhline(1, color="k", lw=0.6, ls=":"); axis.set_xlabel(f"V1-V2 {'eccentricity' if variable == 'ecc' else 'polar-angle'} bin distance")
    axis.set_ylabel("streamlines / expected"); letter(axis, "gh"[column])
    if column == 0: axis.legend(frameon=False, loc="lower center", bbox_to_anchor=(1.1, 1.08), ncol=2)
axis = fig.add_subplot(bottom_grid[2])
for position, pair in enumerate(PAIR_NAMES):
    wide = reference.pivot_table(index="sid", columns="method", values=f"rsi_ree_{pair}")
    for method, shift in (("msmt_csd", -0.15), ("proximity", 0.15)):
        if method in wide: axis.scatter(np.full(len(wide), position + shift), wide[method], s=10, color=colour("reference" if method == "msmt_csd" else method), zorder=3)
    if {"msmt_csd", "proximity"} <= set(wide): axis.plot([position - 0.15, position + 0.15], wide[["msmt_csd", "proximity"]].T, color="0.6", lw=0.5)
axis.axhline(0, color="k", lw=0.6); axis.set_xticks(range(3), PAIR_NAMES); axis.set_ylabel("RSI (visual-field distance)"); letter(axis, "i")
save(fig, "fig2_h1_reference")

## Figure 3: V1–V2 streamlines of one subject, by method
The subject, hemisphere and view of Figure 1d; every method starts from the same seeds. Columns: the reference on
the full data, then classical CSD and the two networks on the main clinical-grade protocol. Top row: each V1–V2
streamline coloured by the eccentricity of its V1 end. Bottom row: coloured by its endpoint error, the visual-field
distance between the pRF centres of its two ends, with the colour scale ending at half of `prf.ecc_max`. Each panel gives the number of streamlines drawn and the
subject's V1–V2 RSI for that method.

In [ ]:
def subject_rsi(variable, protocol, col="rsi_ree_V1-V2"):
    """RSI of the Figure 1 subject for one method (network seeds averaged); NaN if it was not scored."""
    source = reference if protocol == "REF" else classical if variable == "csd" else learned
    values = source[(source.sid.astype(str) == str(sid)) & (source.protocol == protocol) & (source.variant == variable)][col]
    return values.mean() if len(values) else np.nan

columns = [("REF", "msmt_csd", "reference"), (MAIN, "csd", "csd"), (MAIN, "voxel", "voxel"), (MAIN, "path", "path")]
error_max = cfg.prf.ecc_max / 2                                        # top of the endpoint-error colour scale (deg)
rng = np.random.default_rng(cfg.seed)
fig, axes = plt.subplots(2, 4, figsize=(7.2, 4.3)); fig.subplots_adjust(wspace=0.04, hspace=0.08, left=0.05, right=0.9, top=0.92, bottom=0.03)
for column, (protocol, method, variant_name) in enumerate(columns):
    tractograms = tracts(sid, protocol, method); axes[0, column].set_title(STYLE[variant_name][1] + ("" if protocol == "REF" else f", {protocol}"), fontsize=7)
    if tractograms:                                               # the same streamlines in both rows
        tractogram = tractograms[0]; streamlines = PL.streamlines_of(tractogram); lower, higher = tractogram["a"], tractogram["b"]
        drawn = np.flatnonzero((tractogram["pair"] == 12) & key["valid"][lower] & key["valid"][higher] & (anat["hemi"][lower] == hemi_index))
        drawn = rng.permutation(drawn)[:800]
    for row in range(2):
        axis = axes[row, column]; white_view.draw(axis, white_view.colors(), alpha=0.3)
        if not tractograms:
            axis.text(0.5, 0.5, "not run", ha="center", transform=axis.transAxes); continue
        if row == 0:
            values = np.clip((key["ecc"][lower[drawn]] - ecc_low) / (ecc_high - ecc_low), 0, 1); colour_map = "viridis"
        else:
            values = np.clip(np.hypot(key["x"][lower[drawn]] - key["x"][higher[drawn]], key["y"][lower[drawn]] - key["y"][higher[drawn]]) / error_max, 0, 1); colour_map = "RdYlBu_r"
        axis.add_collection(LineCollection([np.c_[streamlines[index] @ white_view.right, streamlines[index][:, 2]] for index in drawn], colors=plt.get_cmap(colour_map)(values), linewidths=0.4))
        if row == 1: axis.text(0.98, 0.98, f"n = {len(drawn)}\nRSI = {subject_rsi(method if protocol == "REF" else variant_name, protocol):.2f}", transform=axis.transAxes,
                          fontsize=6, ha="right", va="top")
for row, (colour_map, value_min, value_max, label) in enumerate((("viridis", ecc_low, ecc_high, "V1-end eccentricity (deg)"), ("RdYlBu_r", 0, error_max, "endpoint error (deg)"))):
    ticks = np.linspace(value_min, value_max, 3).round(2)
    bar = plt.colorbar(plt.cm.ScalarMappable(cmap=colour_map, norm=plt.Normalize(value_min, value_max)), cax=axes[row, 3].inset_axes([1.06, 0.1, 0.05, 0.8]), ticks=ticks)
    bar.set_label(label)
    if row == 1: bar.ax.set_yticklabels([f"{tick:g}" for tick in ticks[:-1]] + [f"≥{ticks[-1]:g}"])
for row in range(2): axes[row, 0].text(-0.1, 1.0, "ab"[row], transform=axes[row, 0].transAxes, fontsize=10, fontweight="bold", va="top")
save(fig, "fig3_bundles")

## Figure 4: path loss against voxel loss (H3)
(a–d) V1–V2 eccentricity ratios on the main clinical-grade protocol: reference, classical CSD, and the two networks
that differ only in the loss. (e) RSI on each clinical-grade protocol, one point per subject, lines from the
voxel-loss to the path-loss network of the same subject; the dashed line is the median reference RSI. (f) Voxel-wise
accuracy against RSI, median and inter-quartile range over subjects, for every method and clinical protocol.

In [ ]:
fig = plt.figure(figsize=(7.2, 6.4)); gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.6], hspace=0.35, wspace=0.75)
for column, (method, protocol_name, label) in enumerate((("msmt_csd", "REF", "reference"), ("csd", MAIN, f"classical CSD, {MAIN}"),
                                    ("voxel", MAIN, f"voxel loss, {MAIN}"), ("path", MAIN, f"path loss, {MAIN}"))):
    axis = fig.add_subplot(gs[0, column]); ratio, _ = pooled(method, protocol_name, 12, "ecc"); image = matrix(axis, ratio, label, "V1-V2", "ecc"); letter(axis, "abcd"[column], -0.45)
    if column: axis.set_ylabel("")
if image is not None: colorbar(fig, image, axis)
bottom_grid = gs[1, :].subgridspec(1, 2, wspace=0.3)
axis = fig.add_subplot(bottom_grid[0]); offset = 0.25; reference_median = reference[reference.method == "msmt_csd"].rsi_ree_all.median()
by_method = pd.concat([classical[classical.method == "csd"], learned[learned.variant.isin(["voxel", "path"])]]).groupby(["sid", "protocol", "variant"]).rsi_ree_all.mean().unstack("variant")
for position, protocol in enumerate(clinical):
    scores = by_method.xs(protocol, level="protocol") if protocol in by_method.index.get_level_values("protocol") else pd.DataFrame()
    for index, method in enumerate(("csd", "voxel", "path")):
        if method in scores: axis.scatter(np.full(len(scores), position + (index - 1) * offset), scores[method], s=10, color=colour(method), zorder=3, label=STYLE[method][1] if position == 0 else None)
    if {"voxel", "path"} <= set(scores): axis.plot([position, position + offset], scores[["voxel", "path"]].T, color="0.6", lw=0.5)
axis.axhline(reference_median, color="k", ls="--", lw=0.8, label="reference, median"); axis.axhline(0, color="k", lw=0.5)
axis.set_xticks(range(len(clinical)), clinical); axis.set_xlim(-0.6, len(clinical) - 0.4); axis.set_xlabel("protocol"); axis.set_ylabel("RSI"); letter(axis, "e")
axis.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.16))
axis = fig.add_subplot(bottom_grid[1]); combined = pd.concat([classical[classical.protocol.isin(clinical)], learned[learned.protocol.isin(clinical)]])
combined = combined.groupby(["sid", "protocol", "variant"])[["acc", "rsi_ree_all"]].mean().reset_index()
order = [method for method in ("csd", "voxel", "path") if method in set(combined.variant)] + sorted(set(combined.variant) - {"csd", "voxel", "path"})
markers = dict(zip(clinical, "os^dv<>"))
for (protocol, method), scores in combined.groupby(["protocol", "variant"]):
    quantiles = scores[["acc", "rsi_ree_all"]].quantile([.25, .5, .75])
    axis.errorbar(quantiles.acc[.5], quantiles.rsi_ree_all[.5], xerr=[[quantiles.acc[.5] - quantiles.acc[.25]], [quantiles.acc[.75] - quantiles.acc[.5]]],
               yerr=[[quantiles.rsi_ree_all[.5] - quantiles.rsi_ree_all[.25]], [quantiles.rsi_ree_all[.75] - quantiles.rsi_ree_all[.5]]],
               fmt=markers[protocol], color=colour(method), ms=4, lw=0.6, capsize=0, zorder=3 if method in ("voxel", "path") else 2)
axis.axhline(reference_median, color="k", ls="--", lw=0.8); axis.set_xlabel("voxel-wise accuracy (ACC)"); axis.set_ylabel("RSI"); letter(axis, "f")
handles = [plt.Line2D([], [], color=colour(method), marker="o", ls="", ms=4, label=STYLE[method][1] if method in STYLE else method) for method in order]
handles += [plt.Line2D([], [], color="0.3", marker=markers[protocol], ls="", ms=4, label=protocol) for protocol in clinical]
axis.legend(handles=handles, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16), columnspacing=0.8, handletextpad=0.2)
save(fig, "fig4_h3_path_vs_voxel")

## Figure 5: recovery of voxel-wise accuracy and of retinotopy (H2)
Recovery = (network − classical CSD) / (reference − classical CSD) on the same protocol, for the voxel-loss network,
as in the H2 test of notebook 09: cells where CSD loses less than 0.02 RSI are left out. (a) One small point per
subject × protocol, one large point per subject (median over protocols); above the line, accuracy recovers more.
(b) The same for the path-loss network, for comparison.

In [ ]:
def recovery(variable):
    csd_scores = classical[classical.method == "csd"].set_index(["sid", "protocol"])[["acc", "rsi_ree_all"]]
    network_scores = learned[learned.variant == variable].groupby(["sid", "protocol"])[["acc", "rsi_ree_all"]].mean()
    joined = network_scores.join(csd_scores, rsuffix="_csd", how="inner").reset_index(); joined = joined[joined.protocol.isin(clinical)]
    joined["rsi_ref"] = joined.sid.map(reference[reference.method == "msmt_csd"].set_index("sid").rsi_ree_all)
    rsi_lost = joined.rsi_ref - joined.rsi_ree_all_csd
    joined["rec_acc"] = (joined.acc - joined.acc_csd) / (1 - joined.acc_csd); joined["rec_rsi"] = ((joined.rsi_ree_all - joined.rsi_ree_all_csd) / rsi_lost).where(rsi_lost >= 0.02)
    joined = joined.replace([np.inf, -np.inf], np.nan); both = joined.rec_acc.notna() & joined.rec_rsi.notna()
    joined[["rec_acc", "rec_rsi"]] = joined[["rec_acc", "rec_rsi"]].where(both)   # same cells as the H2 test
    return joined

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4)); recoveries = {values: recovery(values) for values in ("voxel", "path")}
values = np.r_[[0, 1], *[np.r_[joined.rec_rsi.dropna(), joined.rec_acc.dropna()] for joined in recoveries.values()]]; limits = [values.min() - 0.1 * np.ptp(values), values.max() + 0.1 * np.ptp(values)]
for axis, variant_name, letter_text in zip(axes, ("voxel", "path"), "ab"):
    joined = recoveries[variant_name]; subject_medians = joined.groupby("sid")[["rec_acc", "rec_rsi"]].median()
    axis.scatter(joined.rec_rsi, joined.rec_acc, s=6, color=colour(variant_name), alpha=0.4); axis.scatter(subject_medians.rec_rsi, subject_medians.rec_acc, s=24, color=colour(variant_name), edgecolor="k", lw=0.4)
    axis.plot(limits, limits, color="k", lw=0.6, ls=":"); axis.axhline(0, color="0.8", lw=0.5); axis.axvline(0, color="0.8", lw=0.5)
    axis.set_xlim(limits); axis.set_ylim(limits); axis.set_aspect("equal"); axis.set_xlabel("recovery of RSI"); axis.set_title(STYLE[variant_name][1]); letter(axis, letter_text)
    axis.text(0.98, 0.02, f"{int(joined.rec_rsi.isna().sum())} of {len(joined)} cells without RSI loss", transform=axis.transAxes, ha="right", fontsize=6)
axes[0].set_ylabel("recovery of voxel-wise accuracy"); save(fig, "fig5_h2_recovery")

## Figure 6: other scanners and reproducibility
RSI per subject on (a) HCP 7T and (b) NSD scans, which no network saw in training; lines join the voxel-loss and
path-loss networks of a subject. (c) Agreement of the reference tractogram between two scans of the same person
(HCP test and retest; NSD run 1 and run 2): weighted Dice and the correlation of streamline-end counts per vertex.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8), gridspec_kw=dict(width_ratios=[1, 1, 0.8], wspace=0.45))
for axis, (file, label), letter_text in zip(axes, (("external_7t", "HCP 7T"), ("external_nsd", "NSD")), "ab"):
    scores = table(file)
    if scores.empty: axis.text(0.5, 0.5, "no data", ha="center", transform=axis.transAxes); axis.set_title(label); continue
    wide = scores.groupby(["sid", "variant"]).rsi_ree_all.mean().unstack(); order = [method for method in ("proximity", "csd", "voxel", "path") if method in wide] + sorted(set(wide) - set(STYLE))
    for position, method in enumerate(order): axis.scatter(np.full(len(wide), position), wide[method], s=10, color=colour(method), zorder=3)
    if {"voxel", "path"} <= set(wide): axis.plot([order.index("voxel"), order.index("path")], wide[["voxel", "path"]].T, color="0.6", lw=0.5)
    axis.axhline(0, color="k", lw=0.5); axis.set_xticks(range(len(order)), order, rotation=45, ha="right"); axis.set_ylabel("RSI"); axis.set_title(f"{label} (n = {len(wide)})"); letter(axis, letter_text)
axis = axes[2]; tables = []
for file, label in (("retest_overlap", "HCP retest"), ("nsd_run_overlap", "NSD runs")):
    scores = table(file)
    if not scores.empty: tables.append(scores[scores.method == scores.method.iloc[0]].assign(dataset=label))
if tables:
    scores = pd.concat(tables)
    for position, (column, label) in enumerate((("wdice_retest", "weighted Dice"), ("vertex_count_r", "end-count r"))):
        for index, (dataset, dataset_rows) in enumerate(scores.groupby("dataset", sort=False)):
            axis.scatter(np.full(len(dataset_rows), position + (index - 0.5) * 0.3), dataset_rows[column], s=12, color=f"C{index + 4}", label=dataset if position == 0 else None)
    axis.set_xticks([0, 1], ["weighted Dice", "end-count r"]); axis.set_ylim(0, 1); axis.legend(frameon=False, fontsize=6)
axis.set_title("reproducibility"); letter(axis, "c")
save(fig, "fig6_external_reproducibility")

## Figure 7: networks against classical CSD
Rows: voxel-wise accuracy (ACC), RSI and path error (mm, lower is better); columns: the clinical-grade protocols. One
point per subject for classical CSD and for the voxel-loss and path-loss networks (mean over training seeds); grey
lines join the scores of one subject and black bars are medians. A bracket with * marks a test of the comparison with
classical CSD (`csd_comparison_tests.csv`, notebook 09) whose Holm-corrected p is below `stats.alpha`: ACC path >
CSD; RSI voxel < CSD and path not worse than CSD by more than `stats.rsi_margin`; path error path < CSD and path <
voxel. Without that table the figure has no brackets.

In [ ]:
# Per-subject scores of classical CSD and of the two networks (mean over training seeds); a bracket with * marks a
# comparison of the classical-CSD family whose Holm-corrected p is below alpha.
MEASURES = (("acc", "ACC"), ("rsi_ree_all", "RSI"), ("path_err_mm", "path error (mm)"))
TESTS = {"acc": [("ACC path > classical CSD", 0, 2)], "rsi_ree_all": [("RSI voxel-only < classical CSD", 0, 1), ("RSI path not worse than classical CSD", 0, 2)],
         "path_err_mm": [("path error path < classical CSD", 0, 2), ("path error path < voxel-only", 1, 2)]}
path_error, tests = table("path_error"), table("csd_comparison_tests")
scores = pd.concat([classical[classical.method == "csd"], learned[learned.variant.isin(["voxel", "path"])]])
scores = pd.concat([scores, path_error[path_error.variant.isin(["csd", "voxel", "path"])]]) if not path_error.empty else scores
scores = scores[scores.protocol.isin(clinical)].groupby(["sid", "protocol", "variant"])[[c for c, _ in MEASURES if c in scores]].mean()
fig, axes = plt.subplots(len(MEASURES), len(clinical), figsize=(1.8 * len(clinical) + 0.6, 6.6), squeeze=False, gridspec_kw=dict(hspace=0.45, wspace=0.45))
for row, (column, label) in enumerate(MEASURES):
    for position, protocol in enumerate(clinical):
        axis = axes[row, position]; letter(axis, "abcdefghijklmnopqrstuvwxyz"[row * len(clinical) + position], -0.3)
        wide = scores[column].xs(protocol, level="protocol").unstack("variant") if column in scores and protocol in scores.index.get_level_values("protocol") else pd.DataFrame()
        order = [method for method in ("csd", "voxel", "path") if method in wide and wide[method].notna().any()]
        axis.set_xticks(range(3), ["CSD", "voxel", "path"]); axis.set_xlim(-0.6, 2.6)
        if row == 0: axis.set_title(protocol)
        if position == 0: axis.set_ylabel(label)
        if not order: axis.text(0.5, 0.5, "no data", ha="center", va="center", transform=axis.transAxes, fontsize=7); axis.set_yticks([]); continue
        positions = [("csd", "voxel", "path").index(method) for method in order]
        if len(order) > 1: axis.plot(positions, wide[order].T, color="0.8", lw=0.5, zorder=1)
        for x, method in zip(positions, order):
            axis.scatter(np.full(len(wide), x), wide[method], s=10, color=colour(method), zorder=3)
            axis.plot([x - 0.25, x + 0.25], [wide[method].median()] * 2, color="k", lw=1.2, zorder=4)
        low, high = np.nanmin(wide[order].values), np.nanmax(wide[order].values); span = (high - low) or 1; level = high
        for name, start, end in sorted(TESTS[column], key=lambda test: test[2] - test[1]) if not tests.empty else []:   # short brackets lowest
            row_test = tests[(tests.protocol == protocol) & (tests.test == name)]
            if row_test.empty or not row_test.p_holm.iloc[0] < cfg.stats.alpha: continue
            level += 0.2 * span; axis.plot([start, start, end, end], [level - 0.03 * span, level, level, level - 0.03 * span], color="k", lw=0.6)
            axis.text((start + end) / 2, level, "*", ha="center", va="bottom", fontsize=9)
        axis.set_ylim(low - 0.08 * span, level + 0.2 * span)
handles = [plt.Line2D([], [], color=colour(method), marker="o", ls="", ms=4, label=STYLE[method][1]) for method in ("csd", "voxel", "path")]
fig.legend(handles=handles + [plt.Line2D([], [], color="k", lw=1.2, label="median")], frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 0.06))
save(fig, "fig7_csd_comparison")

## Figure 8: training size
Median paired difference with its bootstrap interval against the number of training subjects (log axis), one line
per clinical-grade protocol: RSI path − voxel (H3), RSI voxel − CSD, RSI path − CSD, path error path − voxel and path
error path − CSD (`training_size_tests.csv`, notebook 09). The size `n_train` of this run is the confirmatory
analysis; the others are the runs in `training_size_runs`. Drawn when the table holds at least two sizes.

In [ ]:
# Median paired difference and its interval against the number of training subjects, one line per protocol.
PANELS = (("RSI path > voxel-only", "RSI\npath − voxel"), ("RSI voxel-only < classical CSD", "RSI\nvoxel − CSD"),
          ("RSI path not worse than classical CSD", "RSI\npath − CSD"), ("path error path < voxel-only", "path error (mm)\npath − voxel"),
          ("path error path < classical CSD", "path error (mm)\npath − CSD"))
by_size = table("training_size_tests"); sizes = sorted(map(int, by_size.n_train.unique())) if not by_size.empty else []
if len(sizes) < 2:
    print(f"training_size_tests.csv holds {len(sizes)} training size(s); Figure 8 needs at least two")
else:
    by_size["comparison"] = by_size.test.str.replace(r", P\w+$", "", regex=True)
    fig, axes = plt.subplots(1, len(PANELS), figsize=(7.2, 2.3), gridspec_kw=dict(wspace=0.6))
    for axis, (name, label), text in zip(axes, PANELS, "abcde"):
        for index, protocol in enumerate(clinical):
            rows = by_size[(by_size.comparison == name) & (by_size.protocol == protocol)].sort_values("n_train")
            if rows.empty: continue
            offset = 1 + (index - (len(clinical) - 1) / 2) * 0.05             # keeps the protocols' bars apart
            error = [np.clip(rows["median"] - rows.ci_lo, 0, None), np.clip(rows.ci_hi - rows["median"], 0, None)]
            axis.errorbar(rows.n_train * offset, rows["median"], yerr=error, color=f"C{index}", marker="o", ms=3, lw=0.8, capsize=2, label=protocol)
        axis.axhline(0, color="k", lw=0.5); axis.set_xscale("log"); axis.set_xticks(sizes, [str(size) for size in sizes]); axis.minorticks_off()
        axis.set_title(label); axis.set_xlabel("training subjects"); letter(axis, text, -0.35)
    axes[0].set_ylabel("median paired difference"); axes[0].legend(frameon=False)
    save(fig, "fig8_training_size")

## Supplementary figures
S1: every learned variant and classical method on the clinical-grade protocols (RSI and voxel-wise accuracy).
S2: reference RSI under the alternative analysis choices of notebook 08. S3: the path-against-voxel contrast scored
with keys that need no functional scan (group-average pRF maps, anatomical atlas). S4: all pairwise comparisons.

In [ ]:
combined = pd.concat([classical[classical.protocol.isin(clinical)], learned[learned.protocol.isin(clinical)]]).groupby(["sid", "protocol", "variant"])[["rsi_ree_all", "acc"]].mean().reset_index()
order = [method for method in ("csd", "voxel", "path") if method in set(combined.variant)] + sorted(set(combined.variant) - {"csd", "voxel", "path"})
fig, axes = plt.subplots(2, 1, figsize=(7.2, 4.6), sharex=True)
for axis, column, label in zip(axes, ("rsi_ree_all", "acc"), ("RSI", "voxel-wise accuracy (ACC)")):
    for position, method in enumerate(order):
        for index, protocol in enumerate(clinical):
            values = combined[(combined.variant == method) & (combined.protocol == protocol)][column].dropna(); shift = (index - (len(clinical) - 1) / 2) * 0.22
            if len(values): axis.boxplot(values, positions=[position + shift], widths=0.18, showfliers=False, patch_artist=True,
                                 boxprops=dict(facecolor=f"C{index}", alpha=0.35, lw=0.5), medianprops=dict(color="k", lw=0.8), whiskerprops=dict(lw=0.5), capprops=dict(lw=0.5))
            axis.scatter(np.full(len(values), position + shift), values, s=4, color=f"C{index}", zorder=3)
    axis.set_ylabel(label)
for index, protocol in enumerate(clinical): axes[0].scatter([], [], color=f"C{index}", s=12, label=protocol)
axes[0].legend(frameon=False, ncol=len(clinical), loc="lower right", bbox_to_anchor=(1, 1.0)); axes[0].axhline(reference[reference.method == "msmt_csd"].rsi_ree_all.median(), color="k", ls="--", lw=0.8)
axes[1].set_xticks(range(len(order)), order, rotation=30, ha="right"); letter(axes[0], "a"); letter(axes[1], "b")
save(fig, "figS1_all_methods")

robustness = table("robustness")
if not robustness.empty:
    fig, axis = plt.subplots(figsize=(7.2, 2.6)); analyses = list(dict.fromkeys(robustness.analysis))
    for position, analysis in enumerate(analyses):
        values = robustness[robustness.analysis == analysis].rsi_ree_all.dropna(); axis.scatter(np.full(len(values), position), values, s=10, color="k" if analysis == "default" else "C0")
        if len(values): axis.hlines(values.median(), position - 0.3, position + 0.3, color="C3", lw=1.2)
    axis.axhline(0, color="k", lw=0.5); axis.set_xticks(range(len(analyses)), analyses, rotation=35, ha="right"); axis.set_ylabel("reference RSI")
    save(fig, "figS2_robustness")

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
for axis, (column, label) in zip(axes, (("rsi_ree_all", "measured pRF"), ("template_rsi_ree_all", "group-average pRF"), ("atlas_rsi_ree_all", "anatomical atlas"))):
    wide = learned[learned.protocol.isin(clinical)].groupby(["sid", "protocol", "variant"])[column].mean().unstack()
    if {"voxel", "path"} <= set(wide):
        for index, protocol in enumerate(clinical):
            if protocol in wide.index.get_level_values("protocol"):
                scores = wide.xs(protocol, level="protocol"); axis.plot([index - 0.12, index + 0.12], scores[["voxel", "path"]].T, color="0.6", lw=0.5)
                axis.scatter(np.full(len(scores), index - 0.12), scores.voxel, s=8, color=colour("voxel")); axis.scatter(np.full(len(scores), index + 0.12), scores.path, s=8, color=colour("path"))
    axis.axhline(0, color="k", lw=0.5); axis.set_xticks(range(len(clinical)), clinical); axis.set_title(f"key: {label}")
axes[0].set_ylabel("RSI")
axes[1].legend(handles=[plt.Line2D([], [], color=colour(method), marker="o", ls="", ms=4, label=STYLE[method][1]) for method in ("voxel", "path")],
             frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
save(fig, "figS3_keys")

### S4: pairwise comparisons
FDR-corrected p-values of the two-sided Wilcoxon test of RSI between every pair of methods on each clinical-grade
protocol (notebook 09, `pairwise_tests.csv`). Green below `stats.alpha`; grey where the test needs more subjects.

In [ ]:
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap
pairwise = table("pairwise_tests")
if not pairwise.empty:
    label = lambda method: f"{method[4:]} (network)" if method.startswith("net_") else method
    green = LinearSegmentedColormap.from_list("p", ["#00441b", "#41ab5d", "#c7e9c0", "#f7fcf5"])
    edges = [0, 0.001, 0.01, cfg.stats.alpha, 1]; norm = BoundaryNorm(edges, green.N)
    protocols = list(dict.fromkeys(pairwise.protocol)); n_protocols = len(protocols)
    fig, axes = plt.subplots(1, n_protocols, figsize=(7.2, 7.2 / n_protocols + 1.2), squeeze=False); axes = axes[0]
    for axis, protocol in zip(axes, protocols):
        scores = pairwise[pairwise.protocol == protocol]; methods = list(dict.fromkeys(list(scores.method_1) + list(scores.method_2))); n_methods = len(methods)
        p_values = np.full((n_methods, n_methods), np.nan)
        for _, row in scores.iterrows(): p_values[methods.index(row.method_2), methods.index(row.method_1)] = row.p_fdr
        lower_triangle = np.tril(np.ones((n_methods, n_methods), bool), -1)
        axis.imshow(np.where(lower_triangle & np.isnan(p_values), 1.0, np.nan), cmap="Greys", vmin=0, vmax=4)          # untested cells
        image = axis.imshow(np.where(lower_triangle, p_values, np.nan), cmap=green, norm=norm)
        for row_index, column_index in zip(*np.nonzero(lower_triangle)):
            if np.isfinite(p_values[row_index, column_index]): axis.text(column_index, row_index, "<.001" if p_values[row_index, column_index] < 0.001 else f"{p_values[row_index, column_index]:.3f}".lstrip("0"), ha="center", va="center",
                                            fontsize=4.5, color="w" if p_values[row_index, column_index] < 0.01 else "k")
        axis.set_xticks(range(n_methods - 1), [label(method) for method in methods[:-1]], rotation=90); axis.set_yticks(range(1, n_methods), [label(method) for method in methods[1:]])
        axis.set_xlim(-0.5, n_methods - 1.5); axis.set_ylim(n_methods - 0.5, 0.5); axis.set_title(protocol); axis.tick_params(length=0, labelsize=5)
        for spine in axis.spines.values(): spine.set_visible(False)
    bar = fig.colorbar(image, ax=list(axes), shrink=0.6, ticks=edges, pad=0.02); bar.set_label("FDR-corrected p"); bar.ax.tick_params(labelsize=5)
    save(fig, "figS4_pairwise")
else:
    print("pairwise_tests.csv not found: run notebook 09 first")